# Causal estimate of funding and subsequent returns

This notebook keeps causal estimation separate from predictive model results. The request resolves
the funding treatment, continuous return outcome, confounders, chronological nuisance folds,
embargo, nuisance estimator parameters, and cadence-aware placebo policy before fitting.

**Learning objectives**

- state the treatment, outcome, and confounders that define the causal estimand;
- inspect chronological nuisance folds and temporal refutation policy; and
- distinguish a causal result from predictive model diagnostics.

**Book reference:** Chapter 15, causal inference for trading research.

**Prerequisites:** finalized funding features, return labels, and purged walk-forward folds.

In [1]:
import os

import polars as pl

from case_studies.crypto_perps_funding.research_workflow import open_study
from case_studies.research import supersedes_for

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE = os.environ.get("ML4T_OUTPUT_DIR", "")
LABEL = "fwd_ret_8h"
CONFIG_NAME = "dml"
PREVIEW_REDUCTIONS = {}
OVERRIDES = {}
# The 2026-08-24 run, whose identity this one retires. A causal identity still hashes the
# whole of case_studies/utils/causal.py, so any edit to that file moves it, and a label
# resolves to exactly one canonical identity - naming the predecessor is how the newer
# run replaces it instead of sitting alongside it.
SUPERSEDES_CAUSAL: str = "4a7d323f9c80"

## Resolve the estimand and refutation contract

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
request = study.causal(
    method="dml",
    label=LABEL,
    config_name=CONFIG_NAME,
    execution_tier=EXECUTION_TIER,
    preview_reductions=PREVIEW_REDUCTIONS,
    overrides=OVERRIDES,
    supersedes=supersedes_for(SUPERSEDES_CAUSAL, LABEL, labels=[LABEL]),
)
resolved = request.resolve()
computation = resolved.spec["computation"]
pl.DataFrame(
    {
        "causal_hash": [resolved.identity],
        "outcome": [computation["estimand"]["outcome"]],
        "treatment": [computation["estimand"]["treatment"]],
        "outcome_horizon": [computation["estimand"]["outcome_horizon"]],
        "n_folds": [computation["cv"]["n_folds"]],
        "embargo_periods": [computation["cv"]["embargo_periods"]],
        "block_size": [computation["refutation"]["block_size"]],
        "block_size_basis": [computation["refutation"]["block_size_basis"]],
        "gap_policy": [computation["refutation"]["temporal_gap_policy"]],
        "eligible_rows": [computation["analysis_population"]["n_rows"]],
    }
)

causal_hash,outcome,treatment,outcome_horizon,n_folds,embargo_periods,block_size,block_size_basis,gap_policy,eligible_rows
str,str,str,str,i64,i64,i64,str,str,i64
"""297a10843a3d""","""fwd_ret_8h""","""premium_zscore_14d""","""0 days 08:00:00""",5,1,42,"""treatment_window""","""reset""",58975


`block_size` is the parameter the refutation lives or dies on, so it is on the
table rather than buried in the spec. The placebo permutes contiguous blocks
within each symbol; a block of one bar is an iid shuffle, which destroys the
serial dependence the placebo is meant to keep and makes the test trivially
easy to pass. Two things create that dependence and the block spans the longer
of them: the overlapping labels span the outcome horizon, and the treatment
spans its own construction window. Here the horizon is a single 8-hour bar
while `premium_zscore_14d` is a 42-bar rolling statistic, so the treatment
window sets the block and `block_size_basis` says so.

## Execute the separate causal result

In [4]:
result = resolved.run()
if not result.complete or result.spec != resolved.spec:
    raise RuntimeError("causal execution is incomplete or has conflicting identity")
pl.DataFrame(
    {
        "causal_hash": [result.hash],
        "n_obs": [result.metrics["n_obs"]],
        "complete": [result.complete],
        "execution_tier": [result.execution_tier],
    }
)

~/ml4t/public-s6-crypto_perps_funding/case_studies/utils/causal.py:1545: UserWarning: block permutation with block_size=42 cannot move 0.8% of the treatment rows: they sit in segments too short to hold two blocks, so the placebo distribution holds them at their observed values and the refutation p-value is biased toward 1. Read placebo_frozen_fraction alongside the p-value, and lower block_size or widen gap_tolerance if the frozen share is large.
  results = run_dml_analysis(


causal_hash,n_obs,complete,execution_tier
str,i64,bool,str
"""297a10843a3d""",54314,true,"""canonical"""


## Key takeaways and limitations

- The causal identity includes the estimand, nuisance models, sample population, folds, and
  refutation settings.
- Preview sample limits remain outside canonical causal results.
- Double machine learning adjusts for declared observed confounders; it cannot remove bias from an
  omitted cause or establish that the identifying assumptions hold.